In [ ]:
#Veri Bilimi Proje Ödevi(Ülkeler)
# Gerekli Kütüphaneler
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, classification_report, confusion_matrix


#Grafik Ayarları
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

#Veri Yükleme ve Ön İşleme
df = pd.read_csv('country.csv', sep=',', decimal=',')

# Eksik Veriler
if 'Unnamed: 20' in df.columns:
    df.drop(columns=['Unnamed: 20'], inplace=True)

df.columns = df.columns.str.strip()
df['Country'] = df['Country'].str.strip()




In [ ]:
import pandas as pd
import numpy as np
import os

# Veri setinin yüklenmesi
try:
    df = pd.read_csv('country.csv', sep=',', decimal=',')

    print("✅ 'country.csv' başarıyla yüklendi!")

    # Veri seti bilgileri
    print(f"\nCountry Dataset:")
    print(f"  - Boyut: {df.shape}")
    print(f"  - Sütunlar: {list(df.columns)}")


except FileNotFoundError as e:
    print(f"⚠️ Veri dosyası bulunamadı: {e}")
    print("🔄 Örnek veri (Mock Data) oluşturuluyor...")

    # Örnek veri oluşturma fonksiyonu
    def create_country_sample_data():
        np.random.seed(42)
        n_samples = 227

        regions = [
            'ASIA (EX. NEAR EAST)', 'EASTERN EUROPE', 'NORTHERN AFRICA',
            'OCEANIA', 'WESTERN EUROPE', 'SUB-SAHARAN AFRICA',
            'LATIN AMER. & CARIB', 'C.W. OF IND. STATES',
            'NEAR EAST', 'NORTHERN AMERICA', 'BALTICS'
        ]

        data = {
            'Country': [f'Country_{i}' for i in range(1, n_samples + 1)],
            'Region': np.random.choice(regions, n_samples),
            'Population': np.random.randint(20000, 1300000000, n_samples),
            'Area (sq. mi.)': np.random.randint(100, 17000000, n_samples),
            'Pop. Density (per sq. mi.)': np.random.uniform(1, 1000, n_samples),
            'Coastline (coast/area ratio)': np.random.uniform(0, 100, n_samples),
            'Net migration': np.random.uniform(-10, 10, n_samples),
            'Infant mortality (per 1000 births)': np.random.uniform(2, 160, n_samples),
            'GDP ($ per capita)': np.random.choice([700, 2500, 6000, 15000, 35000, 55000], n_samples) + np.random.normal(0, 500, n_samples),
            'Literacy (%)': np.random.uniform(30, 100, n_samples),
            'Phones (per 1000)': np.random.uniform(1, 900, n_samples),
            'Arable (%)': np.random.uniform(0, 60, n_samples),
            'Crops (%)': np.random.uniform(0, 40, n_samples),
            'Other (%)': np.random.uniform(0, 40, n_samples),
            'Climate': np.random.randint(1, 5, n_samples),
            'Birthrate': np.random.uniform(8, 50, n_samples),
            'Deathrate': np.random.uniform(5, 30, n_samples),
            'Agriculture': np.random.uniform(0, 0.7, n_samples),
            'Industry': np.random.uniform(0, 0.6, n_samples),
            'Service': np.random.uniform(0, 0.8, n_samples)
        }

        return pd.DataFrame(data)

    # Örnek verileri oluştur
    df = create_country_sample_data()
    print("✅ Örnek veriler oluşturuldu!")

    # (Orijinal formatı taklit etmek için sep=';' ve decimal=',' kullanıyoruz)
    df.to_csv('country.csv', index=False, sep=';', decimal=',')
    print("💾 'country.csv' dosyası kaydedildi.")

# Sonuç olarak temiz bir dataframe elimizde
print("\nVeri analize hazır.")

✅ 'country.csv' başarıyla yüklendi!

Country Dataset:
  - Boyut: (227, 20)
  - Sütunlar: ['Country', 'Region', 'Population', 'Area (sq. mi.)', 'Pop. Density (per sq. mi.)', 'Coastline (coast/area ratio)', 'Net migration', 'Infant mortality (per 1000 births)', 'GDP ($ per capita)', 'Literacy (%)', 'Phones (per 1000)', 'Arable (%)', 'Crops (%)', 'Other (%)', 'Climate', 'Birthrate', 'Deathrate', 'Agriculture', 'Industry', 'Service']

Veri analize hazır.


In [ ]:
#1.SORU
# 'Population' sütununu sayıya çevir
df['Population'] = pd.to_numeric(df['Population'], errors='coerce')

# 1. GÖREV: Nüfusa Göre Azalan Sıralama ---

task1 = df.sort_values(by='Population', ascending=False)

# Sonucu Göster (Sadece Ülke ve Nüfus sütunlarını alalım)
print("En Kalabalık İlk 10 Ülke:")
print(task1[['Country', 'Population']].head(10).to_string(index=False))


En Kalabalık İlk 10 Ülke:
      Country  Population
        China  1313973713
        India  1095351995
United States   298444215
    Indonesia   245452739
       Brazil   188078227
     Pakistan   165803560
   Bangladesh   147365352
       Russia   142893540
      Nigeria   131859731
        Japan   127463611


In [ ]:
#2.SORU
# Sütun ismini daha kolay kullanım için değiştirelim
df.rename(columns={'GDP ($ per capita)': 'GDP'}, inplace=True)

task2 = df.sort_values(by='GDP', ascending=True)

print("Kişi Başına Düşen Hasıla :")
print(task2[['Country', 'GDP']].head(227).to_string(index=False))


Kişi Başına Düşen Hasıla :
                         Country     GDP
                      East Timor   500.0
                    Sierra Leone   500.0
                         Somalia   500.0
                         Burundi   600.0
                          Malawi   600.0
                        Tanzania   600.0
                      Gaza Strip   600.0
                         Comoros   700.0
                Congo, Dem. Rep.   700.0
                        Ethiopia   700.0
                     Afghanistan   700.0
            Congo, Repub. of the   700.0
                         Eritrea   700.0
                      Madagascar   800.0
                   Guinea-Bissau   800.0
                        Kiribati   800.0
                           Niger   800.0
                           Yemen   800.0
                       West Bank   800.0
                          Zambia   800.0
                            Mali   900.0
                         Nigeria   900.0
                      Tajikist

In [ ]:

# 3.SORU
# Population sütununu sayıya çevir
df['Population'] = pd.to_numeric(df['Population'], errors='coerce')

# Sadece Population değeri 10.000.000'dan büyük olan satırları seç
task3 = df[df['Population'] > 10000000]

# Kaç tane olduğunu da ekrana yazalım
print(f"Nüfusu 10 Milyonun Üzerinde Olan Ülke Sayısı: {len(task3)}")
print(task3[['Country', 'Population']].head(79).to_string(index=False))

Nüfusu 10 Milyonun Üzerinde Olan Ülke Sayısı: 79
         Country  Population
     Afghanistan    31056997
         Algeria    32930091
          Angola    12127071
       Argentina    39921833
       Australia    20264082
      Bangladesh   147365352
         Belarus    10293011
         Belgium    10379067
          Brazil   188078227
    Burkina Faso    13902972
           Burma    47382633
        Cambodia    13881427
        Cameroon    17340702
          Canada    33098932
           Chile    16134219
           China  1313973713
        Colombia    43593035
Congo, Dem. Rep.    62660551
   Cote d'Ivoire    17654843
            Cuba    11382820
  Czech Republic    10235455
         Ecuador    13547510
           Egypt    78887007
        Ethiopia    74777981
          France    60876136
         Germany    82422299
           Ghana    22409572
          Greece    10688058
       Guatemala    12293545
           India  1095351995
       Indonesia   245452739
            Iran    686

In [ ]:
#4.SORU

# 'Literacy (%)' sütununu sayıya çevir (Hata almamak için)
df['Literacy (%)'] = pd.to_numeric(df['Literacy (%)'], errors='coerce')

# ascending=False -> Büyükten küçüğe
task4 = df.sort_values(by='Literacy (%)', ascending=False)

# İlk 5 ülkeyi seç (Sadece Ülke ve Okuryazarlık sütunlarını al)
top5_literacy = task4[['Country', 'Literacy (%)']].head(5)

print("En Yüksek Okur-Yazarlık Oranına Sahip İlk 5 Ülke:")
print(top5_literacy.to_string(index=False))

En Yüksek Okur-Yazarlık Oranına Sahip İlk 5 Ülke:
   Country  Literacy (%)
   Andorra         100.0
 Australia         100.0
   Finland         100.0
   Denmark         100.0
Luxembourg         100.0


In [ ]:
#5.SORU

df.rename(columns={'GDP ($ per capita)': 'GDP'}, inplace=True)

# Sadece GDP değeri 10000'den büyük olanları seç
task5 = df[df['GDP'] > 10000]

# Sonucu Göster
# Hem sayısını hem de listeyi yazdıralım
print(f"Kişi Başı Geliri 10.000$'dan Yüksek Olan Ülke Sayısı: {len(task5)}")
print(task5[['Country', 'GDP']].head(10).to_string(index=False)) # İlk 10 tanesini örnek olarak göster

Kişi Başı Geliri 10.000$'dan Yüksek Olan Ülke Sayısı: 76
          Country     GDP
          Andorra 19000.0
Antigua & Barbuda 11000.0
        Argentina 11200.0
            Aruba 28000.0
        Australia 29000.0
          Austria 30000.0
     Bahamas, The 16700.0
          Bahrain 16900.0
         Barbados 15700.0
          Belgium 29100.0


In [ ]:
import pandas as pd

# Sütun ismini kısaltalım (Kod yazarken kolaylık sağlar)
df.rename(columns={'Pop. Density (per sq. mi.)': 'Pop_Density'}, inplace=True)

# ascending=False -> Büyükten küçüğe
task6 = df.sort_values(by='Pop_Density', ascending=False)

# İlk 10 ülkeyi seç (Ülke ve Yoğunluk sütunlarını alalım)
top10_density = task6[['Country', 'Pop_Density']].head(10)

print("En Yüksek Nüfus Yoğunluğuna Sahip İlk 10 Ülke:")
print(top10_density.to_string(index=False))

En Yüksek Nüfus Yoğunluğuna Sahip İlk 10 Ülke:
   Country  Pop_Density
    Monaco      16271.5
     Macau      16183.0
 Singapore       6482.2
 Hong Kong       6355.7
 Gibraltar       3989.7
Gaza Strip       3968.8
     Malta       1266.5
   Bermuda       1241.0
  Maldives       1196.7
   Bahrain       1050.5
